In [0]:
%python
dbutils.widgets.text("catalog", "tpca")
dbutils.widgets.text("pipeline_runID", "interactive")
catalog = dbutils.widgets.get("catalog")
pipeline_runID = dbutils.widgets.get("pipeline_runID")

In [0]:
%sql
CREATE OR REPLACE TABLE tpca.bronze.region AS
SELECT *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingested_at,
    :pipeline_runID AS pipeline_runID
 FROM read_files("/Volumes/tpca/bronze/raw/region/", format => 'csv', header => 'true')

In [0]:
%sql
CREATE OR REPLACE TABLE tpca.bronze.nation AS
SELECT *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingested_at,
    :pipeline_runID AS pipeline_runID
 FROM read_files("/Volumes/tpca/bronze/raw/nation/", format => 'csv', header => 'true')

In [0]:
pipeline_runID = dbutils.widgets.get("pipeline_runID")
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.airline USING delta COMMENT 'Airlines'")
spark.sql(f"""
COPY INTO {catalog}.bronze.airline
FROM (
    SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingested_at, :pipeline_runID AS pipeline_runID
    FROM '/Volumes/tpca/bronze/raw/airline/'
)
FILEFORMAT = csv
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""", args={"pipeline_runID": pipeline_runID})

In [0]:
pipeline_runID = dbutils.widgets.get("pipeline_runID")
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.airline_route USING delta COMMENT 'Airline Routes'")
spark.sql(f"""
COPY INTO {catalog}.bronze.airline_route
FROM (
    SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingested_at, :pipeline_runID AS pipeline_runID
    FROM '/Volumes/tpca/bronze/raw/airline_route/'
)
FILEFORMAT = csv
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""", args={"pipeline_runID": pipeline_runID})

In [0]:
pipeline_runID = dbutils.widgets.get("pipeline_runID")
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.route USING delta COMMENT 'Routes'")
spark.sql(f"""
COPY INTO {catalog}.bronze.route
FROM (
    SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingested_at, :pipeline_runID AS pipeline_runID
    FROM '/Volumes/tpca/bronze/raw/route/'
)
FILEFORMAT = csv
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""", args={"pipeline_runID": pipeline_runID})

In [0]:
pipeline_runID = dbutils.widgets.get("pipeline_runID")
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.passenger USING delta COMMENT 'Passengers'")
spark.sql(f"""
COPY INTO {catalog}.bronze.passenger
FROM (
    SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingested_at, :pipeline_runID AS pipeline_runID
    FROM '/Volumes/tpca/bronze/raw/passenger/'
)
FILEFORMAT = csv
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""", args={"pipeline_runID": pipeline_runID})

In [0]:
# Create the target Bronze table
source_path = f"/Volumes/{catalog}/bronze/raw/booking/"
target_table = f"{catalog}.bronze.booking"
checkpoint_path = f"/Volumes/{catalog}/bronze/pipeline_state/checkpoints/booking_v2"
schema_path = f"/Volumes/{catalog}/bronze/pipeline_state/schema/booking"

In [0]:
# Read files 
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

In [0]:
# Add metadata columns for file path and ingestion timestamp
from pyspark.sql.functions import col, current_timestamp, lit

df_stream = (
    df_stream
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("pipeline_runID", lit(pipeline_runID))
)

# Write to Bronze Delta table
query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

In [0]:
# Create the target Bronze table
source_path = f"/Volumes/{catalog}/bronze/raw/flight_booking/"
target_table = f"{catalog}.bronze.flight_booking"
checkpoint_path = f"/Volumes/{catalog}/bronze/pipeline_state/checkpoints/flight_booking"
schema_path = f"/Volumes/{catalog}/bronze/pipeline_state/schema/flight_booking"

In [0]:
# Read files 
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

In [0]:
# Add metadata columns for file path and ingestion timestamp
from pyspark.sql.functions import col, current_timestamp, lit

df_stream = (
    df_stream
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("pipeline_runID", lit(pipeline_runID))
)

# Write to Bronze Delta table
query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

In [0]:
# Count rows for all Bronze tables
bronze_tables = spark.sql(f"SHOW TABLES IN {catalog}.bronze").collect()

print(f"Row counts for {catalog}.bronze:\n")
for t in bronze_tables:
    table_name = t.tableName
    row_count = spark.table(f"{catalog}.bronze.{table_name}").count()
    print(f"{table_name}: {row_count:,} rows")


In [0]:
%sql
select * from tpca.bronze.airline limit 1